# HF_TB Model

## Imports

In [1]:
import LLMs, TP2
from LLMs import *
from TP2 import Dataset

C:\Users\Luco1421\Desktop\U\ia\TPs\TP2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Luco1421\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Charge model

In [2]:
LLMs.clean_gpu()

hugging_face_tb = LLMs.charge_model("HuggingFaceTB/SmolLM3-3B")

Loading weights: 100%|██████████| 326/326 [00:00<00:00, 6143.98it/s]
C:\Users\Luco1421\Desktop\U\ia\TPs\TP2\.venv\Lib\site-packages\torch\nn\modules\module.py:1370: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  return t.to(


## Function to request HF_TB's answer

In [3]:
def chat_hf_tb(text: str, prompt: str):
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": text},
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.1,
            top_p=0.1,
            do_sample=True
        )

    response_text = tokenizer.decode(outputs[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

    LLMs.clean_gpu()

    return extract_results(response_text)

### Example of use

In [5]:
tokenizer, model = hugging_face_tb
chat_hf_tb(SAMPLE_1 + SAMPLE_2 + SAMPLE_3, CONTEXT + TEXT_BEGINNER)

[1, 0, 1]

# Results

## Load dataset and test utils class

In [6]:
dataset = TP2.Dataset("FEINA_1.xlsx").read()
llm_utils = LLMs.TestLLMUtils()

# Test without Few Shots

In [17]:
llm_utils.test_without_shots("Hugging Face TB", chat_hf_tb, dataset)

KeyboardInterrupt: 

# Test with Few Shots

In [18]:
llm_utils.test_with_shots("Hugging Face TB", chat_hf_tb, dataset, [2, 4, 7])

Hugging Face TB with 2 shots - Accuracy: 0.49706518381217174
Hugging Face TB with 4 shots - Accuracy: 0.4851085310449268
Hugging Face TB with 7 shots - Accuracy: 0.48815086782376504


In [7]:
llm_utils.test_all_LLM("Hugging Face TB", chat_hf_tb, dataset, [2, 4, 7])

Hugging Face TB without shots: average = 0.4976, std = 0.0141
Hugging Face TB with 2 shots - average = 0.4983, std = 0.0142
Hugging Face TB with 4 shots - average = 0.4988, std = 0.0152
Hugging Face TB with 7 shots - average = 0.4999, std = 0.0122
--------------------------------------------------


{0: [0.488905325443787,
  0.5150602409638554,
  0.48628048780487804,
  0.4809451219512195,
  0.5129573170731707,
  0.4880774962742176,
  0.5069801616458487,
  0.4958615500376223,
  0.50187265917603,
  0.5143072289156626,
  0.5038402457757296,
  0.48153730218538054,
  0.5195783132530121,
  0.5123226288274833,
  0.4954819277108434,
  0.4771341463414634,
  0.49776119402985075,
  0.5193452380952381,
  0.48134044173648133,
  0.4713855421686747,
  0.49255952380952384,
  0.5003717472118959,
  0.5,
  0.5191873589164786,
  0.4915514592933948,
  0.5163249810174639,
  0.4992378048780488,
  0.49276466108149275,
  0.48698884758364314,
  0.4784905660377359],
 2: [0.4948224852071006,
  0.49623493975903615,
  0.5022865853658537,
  0.504950495049505,
  0.4969512195121951,
  0.5216095380029806,
  0.4996326230712711,
  0.510158013544018,
  0.49887640449438203,
  0.5015060240963856,
  0.5084485407066052,
  0.4777694046721929,
  0.4894578313253012,
  0.5093353248693054,
  0.49849397590361444,
  0.463414634

In [8]:
LLMs.clean_gpu()

del hugging_face_tb, chat_hf_tb, tokenizer, model
gc.collect()

2063